In [1]:
import os
import json
import glob
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================
# 1. CONFIGURATION
# ==========================================
BASE_PATH = "/pscratch/sd/t/tihsu/database/Grid_Study_CMS_OpenData_bbWW_HWW/method_arxiv"
#BASE_PATH = "/pscratch/sd/t/tihsu/database/Grid_Study_CMS_OpenData_v2/method"
JSON_GRID_PATH = "config/sample_list_bbWW_v2.json" 
CUTFLOW_PATH = "/pscratch/sd/t/tihsu/database/Grid_Study_CMS_OpenData_bbWW_HWW/data/cutflow.json"

# ==========================================
# 2. DATA LOADER
# ==========================================

def load_reference_grid(json_path):
    mx_set, my_set = set(), set()
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            config = json.load(f)
        for mass_set in config.get("signal", []):
            match = re.search(r"MX-([\d\.]+)_MY-([\d\.]+)", mass_set)
            if int(float(match.group(2))) in [190, 350, 450, 500]:
                continue
            if match:
                mx_set.add(int(float(match.group(1))))
                my_set.add(int(float(match.group(2))))
    return sorted(list(mx_set)), sorted(list(my_set))

def parse_mass_robust(text):
    match = re.search(r"MX-([\d\.]+).*?_MY-([\d\.]+)", text)
    if match:
        try:
            return int(float(match.group(1))), int(float(match.group(2)))
        except: return None, None
    return None, None

def load_cutflow_data(path):
    records = []
    if not os.path.exists(path):
        return pd.DataFrame()
    try:
        with open(path, 'r') as f:
            cutflow = json.load(f)
            for name, info in cutflow.items():
                if "MX-" not in name: continue
                mx, my = parse_mass_robust(name)
                if mx is not None:
                    records.append({"MX": mx, "MY": my, "passed":float(info["passed"])/2, "eff": float(info["passed"]/info["total"])})
    except Exception as e:
        print(f"Error parsing cutflow: {e}")
    return pd.DataFrame(records)

def load_grid_data(base_path):
    records = []
    def process_file(json_path, method, train_type):
        try:
            filename = os.path.basename(json_path)
            mx, my = parse_mass_robust(filename)
            if mx is None: mx, my = parse_mass_robust(os.path.dirname(json_path).split(os.sep)[-1])
            if mx is None: return
            if "eval_metric" not in json_path: return
            with open(json_path, 'r') as f: data = json.load(f)
            auc_val = data.get("auc", np.nan)
            if pd.isna(auc_val): return
            
            dirname = os.path.dirname(json_path)
            png_candidates = glob.glob(os.path.join(dirname, "score_uniform_binning*.png"))
            specific = [p for p in png_candidates if f"{mx}" in p and f"{my}" in p and "score" in p]
            png_path = specific[0] if specific else next((p for p in png_candidates if "score" in p), None)

            records.append({
                "Method": method, "Type": train_type, "ID": f"{method} ({train_type})",
                "MX": int(mx), "MY": int(my),
                "auc": auc_val, "max_sic": data.get("max_sic", np.nan),
                "trafo_bin_sig": data.get("trafo_bin_sig", np.nan),
                "time": data.get("fitting_time", np.nan), "png_path": png_path
            })
        except: return

    # Standardize path finding
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith(".json") and "eval_metric" in file:
                full_path = os.path.join(root, file)
                if "individual" in full_path:
                    process_file(full_path, full_path.split(os.sep)[-4], "individual")
                elif "parameterized" in full_path and "reduce" not in full_path:
                    process_file(full_path, full_path.split(os.sep)[-4], "parameterized")
                elif "parametrized_reduce" in full_path:
                    try:
                        parts = full_path.split(os.sep)
                        method = parts[-4]
                        cfg = parts[-3]
                        mat = re.search(r"factor_x_(\d+)_y_(\d+)", cfg)
                        # if "xgb" in method.lower():
                        #     continue
                        if mat: process_file(full_path, method, f"sparse_x{mat.group(1)}y{mat.group(2)}")
                        else: process_file(full_path, method, "sparse_param")
                    except: continue
    return pd.DataFrame(records)

# ==========================================
# 3. FANCY PLOTTING FUNCTION (FINAL)
# ==========================================
def plot_comparison_fancy(df, df_cutflow, target_mx, target_my, all_mx, all_my, metric='trafo_bin_sig', baseline_id=None):
    """
    Comparison plot with:
    - Corrected baseline logic (respects dropdown).
    - Headers pinned to top frame.
    - 'Fraction > Baseline' statistic elegantly placed at the bottom.
    """
    # 1. Baseline & Ratios
    real_baseline_id = baseline_id if baseline_id else df['ID'].iloc[0]
    
    if real_baseline_id not in df['ID'].values:
        print(f"Warning: Baseline '{real_baseline_id}' not found. Defaulting to first method.")
        real_baseline_id = df['ID'].iloc[0]

    base_df = df[df['ID'] == real_baseline_id].set_index(['MX', 'MY'])[metric]
    df_merged = df.join(base_df, on=['MX', 'MY'], rsuffix='_base')
    df_merged['ratio'] = df_merged[metric] / df_merged[f'{metric}_base']
    df_merged['ratio'] = df_merged['ratio']

    # Merge Passed Events for Color
    if not df_cutflow.empty:
        df_merged = pd.merge(df_merged, df_cutflow[['MX', 'MY', 'passed']], on=['MX', 'MY'], how='left')
    else:
        df_merged['passed'] = 1

    # 2. Logic & Grouping
    plot_data = []
    models = sorted(df_merged['Method'].unique(), key=lambda x: (0 if 'xgboost' in x.lower() else 1, x))
    
    for model in models:
        m_rows = df_merged[df_merged['Method'] == model]
        modes = sorted(m_rows['Type'].unique())
        
        group_modes = []
        for mode in modes:
            mode_rows = m_rows[m_rows['Type'] == mode]
            
            # --- Labeling Logic ---
            label = mode
            step_x, step_y = 1, 1
            
            if 'individual' in mode.lower():
                label = "Individual"
            elif 'x1y1' in mode.lower():
                label = "Param(Full)"
            else:
                match = re.search(r"x(\d+)y(\d+)", mode)
                if match:
                    step_x, step_y = int(match.group(1)), int(match.group(2))
                    if step_x == 2 and step_y == 2: 
                        label = "Param (1/4)"
                    else: 
                        label = f"Param (1/{step_x*step_y})"
                elif 'param' in mode.lower():
                    label = "Param"

            # Determine Trained vs Interpolated
            points_data = []
            for _, r in mode_rows.iterrows():
                try:
                    idx_x = all_mx.index(r['MX'])
                    idx_y = all_my.index(r['MY'])
                    is_trained = (idx_x % step_x == 0) and (idx_y % step_y == 0)
                except ValueError:
                    is_trained = False
                
                points_data.append({
                    'y': r['ratio'],
                    'z': r['passed'],
                    'is_trained': is_trained
                })

            if "(1/4)" in label or "(1/9)" in label:
                continue

            group_modes.append({'clean_label': label, 'data': points_data})
        plot_data.append({'model': model, 'modes': group_modes})

    # 3. Plotting
    fig, ax = plt.subplots(figsize=(14, 7), dpi=300) # Slightly wider for elegance
    cmap = plt.cm.RdBu 
    norm = plt.Normalize(vmin=df_merged['passed'].min(), vmax=df_merged['passed'].max())
    
    x_cursor = 0
    x_ticks, x_labels = [], []
    jitter_width = 0.2

    for group in plot_data:
        model_name = group['model']
        start_x = x_cursor
        
        for m_obj in group['modes']:
            data = m_obj['data']
            label = m_obj['clean_label']
            
            if data:
                y_vals = np.array([d['y'] for d in data])
                z_vals = np.array([d['z'] for d in data])
                trained_mask = np.array([d['is_trained'] for d in data])
                
                # --- STATS CALCULATION ---
                # Calculate fraction better than baseline (> 1.0)
                n_total = len(y_vals)
                if n_total > 0:
                    frac_better = np.sum(y_vals > 1.0) / n_total
                    stat_text = f"{frac_better:.1%} > Base."

                    stat_color = '#4A6C44' if frac_better >= 0.5 else '#A35648'
                else:
                    stat_text = "N/A"
                    stat_color = 'gray'

                # Jitter X
                noise = np.random.uniform(-jitter_width, jitter_width, size=len(y_vals))
                x_vals = x_cursor + noise
                
                # Plot Points
                if np.any(trained_mask):
                    ax.scatter(x_vals[trained_mask], y_vals[trained_mask], c=z_vals[trained_mask], 
                               cmap=cmap, norm=norm, s=40, marker='D', edgecolors='k', linewidth=0.5, zorder=3)
                if np.any(~trained_mask):
                    ax.scatter(x_vals[~trained_mask], y_vals[~trained_mask], c=z_vals[~trained_mask], 
                               cmap=cmap, norm=norm, s=25, marker='o', edgecolors='k', linewidth=0.3, alpha=0.6, zorder=2)
                
                # --- ADD STAT TEXT ---
                # Placing it at y=0.1 (Data Coordinates) uses the empty bottom space effectively
                ax.text(x_cursor, 0.1, stat_text, 
                        ha='center', va='center', fontsize=6,fontweight='bold', 
                        color=stat_color, bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))

            x_ticks.append(x_cursor)
            x_labels.append(label)
            x_cursor += 1
            
        # Headers (Model Names)
        end_x = x_cursor - 1
        center_x = (start_x + end_x) / 2
        
        # Header text relative to frame (always just above top axis)
        ax.text(center_x, 1.02, model_name.upper(), 
                ha='center', va='bottom', fontweight='bold', fontsize=12, color='#333',
                transform=ax.get_xaxis_transform()) 
        
        ax.axvline(x=x_cursor - 0.5, color='#eee', linestyle='-', linewidth=1.5, zorder=0)
        x_cursor += 0.5

    # 4. Legend & Style
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, alpha=0.5, zorder=1)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel(f"Ratio to Baseline ({metric})", fontsize=11)
    
    # Clean spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False) # Optional: cleaner look
    ax.tick_params(axis='y', length=0)   # Optional: remove y-ticks for cleaner look
    ax.grid(axis='y', linestyle=':', alpha=0.4, zorder=0) # Add light grid
    
    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, aspect=30, pad=0.01)
    cbar.set_label('Passed Events (Sample Size)', rotation=270, labelpad=15)

    plt.ylim(0.0, 3.0) # User specified range

    # Legend
    legend_handles = [
        mlines.Line2D([], [], color='white', marker='D', markerfacecolor='gray', markeredgecolor='k', markersize=8, label='Trained Point'),
        mlines.Line2D([], [], color='white', marker='o', markerfacecolor='gray', markeredgecolor='k', markersize=8, label='Interpolated')
    ]
    ax.legend(handles=legend_handles, loc='upper left', frameon=False, fontsize=10, bbox_to_anchor=(0, 1.0))

    plt.tight_layout()
    plt.savefig("plots/summary.png")
    plt.show()

# ==========================================
# 4. DASHBOARD CLASS
# ==========================================

class GridDashboard:
    def __init__(self):
        self.df = pd.DataFrame()
        self.df_cutflow = pd.DataFrame()
        self.expected_mx, self.expected_my = [], []
        
        self.btn_load = widgets.Button(description="Refresh", button_style='primary')
        self.dd_metric = widgets.Dropdown(options=[('AUC', 'auc'), ('Max SIC', 'max_sic'), ('Significance', 'trafo_bin_sig')], value='trafo_bin_sig', description='Metric:')
        self.dd_baseline = widgets.Dropdown(description="Baseline:")
        self.dd_mx = widgets.Dropdown(description="MX:")
        self.dd_my = widgets.Dropdown(description="MY:")
        self.out_plot = widgets.Output()
        
        self.btn_load.on_click(self.load_data)
        self.dd_metric.observe(self.render, names='value')
        self.dd_baseline.observe(self.render, names='value')
        self.dd_mx.observe(self.render, names='value')
        
        self.ui = widgets.VBox([
            widgets.HBox([self.btn_load, self.dd_metric, self.dd_baseline]),
            widgets.HBox([self.dd_mx, self.dd_my]),
            self.out_plot
        ])
        self.load_data(None)

    def load_data(self, b):
        ref_mx, ref_my = load_reference_grid(JSON_GRID_PATH)
        self.df_cutflow = load_cutflow_data(CUTFLOW_PATH)
        self.df = load_grid_data(BASE_PATH)
        
        if not self.df.empty:
            self.df = self.df.sort_values('auc', ascending=False).drop_duplicates(subset=['ID','MX','MY'])
            self.expected_mx = sorted(list(set(ref_mx)|set(self.df['MX'])))
            self.expected_my = sorted(list(set(ref_my)|set(self.df['MY'])))
            
            methods = sorted(self.df['ID'].unique())
            self.dd_baseline.options = methods
            if methods: self.dd_baseline.value = methods[0]
            self.dd_mx.options = self.expected_mx
            self.dd_my.options = self.expected_my
            
        self.render(None)

    def render(self, b):
        with self.out_plot:
            clear_output(wait=True)
            if self.df.empty: return
            
            mx, my = self.dd_mx.value, self.dd_my.value
            
            plot_comparison_fancy(
                self.df, self.df_cutflow, 
                mx, my,
                self.expected_mx, self.expected_my,
                metric=self.dd_metric.value,
                baseline_id=self.dd_baseline.value
            )

d = GridDashboard()
display(d.ui)

In [21]:
import os
import json
import glob
import re
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
import concurrent.futures

# === IMPORT TQDM ===
from tqdm.notebook import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
BASE_PATH = "/pscratch/sd/t/tihsu/database/Grid_Study_CMS_OpenData_bbWW_HWW/method_arxiv"
#BASE_PATH = "/pscratch/sd/t/tihsu/database/Grid_Study_CMS_OpenData_v2/method"
JSON_GRID_PATH = "config/sample_list_bbWW_v2.json" 
CUTFLOW_PATH = "/pscratch/sd/t/tihsu/database/Grid_Study_CMS_OpenData_bbWW_HWW/data/cutflow.json"
PLOT_DIR = "plots"

# ==========================================
# 2. WORKER FUNCTIONS
# ==========================================

def parse_mass_robust(text):
    match = re.search(r"MX-([\d\.]+).*?_MY-([\d\.]+)", text)
    if match:
        try:
            # if int(float(match.group(2))) in [190, 350, 450, 500]:
            #     return None, None
            return int(float(match.group(1))), int(float(match.group(2)))
        except: return None, None
    return None, None

def worker_process_json(args):
    """Worker to process a single JSON file."""
    json_path, method, train_type = args
    try:
        filename = os.path.basename(json_path)
        mx, my = parse_mass_robust(filename)
        if mx is None: 
            mx, my = parse_mass_robust(os.path.dirname(json_path).split(os.sep)[-1])
        if mx is None: return None
        if "eval_metric" not in json_path: return None

        with open(json_path, 'r') as f: data = json.load(f)
        auc_val = data.get("auc", np.nan)
        if pd.isna(auc_val): return None

        dirname = os.path.dirname(json_path)
        png_candidates = glob.glob(os.path.join(dirname, "score_uniform_binning*.png"))
        specific = [p for p in png_candidates if f"{mx}" in p and f"{my}" in p and "score" in p]
        png_path = specific[0] if specific else next((p for p in png_candidates if "score" in p), None)

        return {
            "Method": method, "Type": train_type, "ID": f"{method} ({train_type})",
            "MX": int(mx), "MY": int(my),
            "auc": auc_val, "time": data.get("fitting_time", np.nan),
            "max_sic": data.get("max_sic", np.nan), "trafo_bin_sig": data.get("trafo_bin_sig", 0.0),
            "png_path": png_path
        }
    except: return None

def worker_save_plot(args):
    """Worker to generate and save a plot."""
    (m, base_name, metric, p_curr, p_base, g_min, g_max, 
     compare_mode, show_text, prec, plot_dir) = args

    plt.switch_backend('Agg') 
    safe_name = m.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "-")
    
    # === CRITICAL FIX: Handle NaNs correctly ===
    # Create a copy of the colormap and set 'bad' (NaN) values to a neutral color
    cmap_val = plt.cm.PuBu.copy()
    cmap_val.set_bad(color='white')
    
    cmap_diff = plt.cm.RdBu_r.copy()
    cmap_diff.set_bad(color='white') # Light grey for missing in ratio plots

    # 1. Absolute Value Plot
    fig1, ax1 = plt.subplots(figsize=(8, 5), dpi=100)
    sns.heatmap(p_curr, ax=ax1, annot=show_text, fmt=f".{prec}f", cmap=cmap_val, vmin=g_min, vmax=g_max, cbar=True)
    ax1.set_title(f"{m}\n({metric})")
    ax1.set_ylabel("MY"); ax1.set_xlabel("MX")
    plt.tight_layout()
    fig1.savefig(f"{plot_dir}/{safe_name}_val.png")
    plt.close(fig1)
    
    # 2. Comparison Plot
    fig2, ax2 = plt.subplots(figsize=(8, 5), dpi=100)
    if compare_mode == 'Ratio':
        # NO fillna(0) HERE!
        ratio = (p_curr / p_base)
        sns.heatmap(ratio, ax=ax2, annot=show_text, fmt=f".{prec}f", cmap=cmap_diff, center=1, vmin=0.5, vmax=1.5)
        ax2.set_title(f"{m} / {base_name}\n(Ratio)")
    else:
        # NO fillna(0) HERE!
        diff = p_curr - p_base
        sns.heatmap(diff, ax=ax2, annot=show_text, fmt=".2f", cmap=cmap_diff, center=0)
        ax2.set_title(f"{m} - {base_name}\n(Diff)")
    
    ax2.set_ylabel("MY"); ax2.set_xlabel("MX")
    plt.tight_layout()
    fig2.savefig(f"{plot_dir}/{safe_name}_compare.png")
    plt.close(fig2)
    
    return m

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================

def load_reference_grid(json_path):
    mx_set, my_set = set(), set()
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            config = json.load(f)
        for mass_set in config.get("signal", []):
            match = re.search(r"MX-([\d\.]+)_MY-([\d\.]+)", mass_set)
            if match: mx_set.add(int(float(match.group(1)))); my_set.add(int(float(match.group(2))))
    return sorted(list(mx_set)), sorted(list(my_set))

def load_cutflow_data(path):
    records = []
    if not os.path.exists(path):
        return pd.DataFrame()
    try:
        with open(path, 'r') as f:
            cutflow = json.load(f)
            for name, info in cutflow.items():
                if "MX-" not in name: continue
                mx, my = parse_mass_robust(name)
                if mx is not None:
                    records.append({"MX": mx, "MY": my, "passed":float(info["passed"]), "eff": float(info["passed"]/info["total"])})
    except Exception as e:
        print(f"Error parsing cutflow: {e}")
    return pd.DataFrame(records)

def plot_comparison_fancy(df, df_cutflow, target_mx, target_my, all_mx, all_my, metric='max_sic', baseline_id='xgboost (individual)'):
    base_candidates = [m for m in df['ID'].unique() if 'xgboost' in m.lower() and 'individual' in m.lower()]
    real_baseline_id = base_candidates[0] if base_candidates else df['ID'].iloc[0]
    base_df = df[df['ID'] == real_baseline_id].set_index(['MX', 'MY'])[metric]
    df_merged = df.join(base_df, on=['MX', 'MY'], rsuffix='_base')
    
    # FIX: No fillna(0)
    df_merged['ratio'] = (df_merged[metric] / df_merged[f'{metric}_base'])
    
    if not df_cutflow.empty:
        df_merged = pd.merge(df_merged, df_cutflow[['MX', 'MY', 'passed']], on=['MX', 'MY'], how='left')
        df_merged['passed'] = df_merged['passed'].fillna(0)
    else: df_merged['passed'] = 1 

    plot_data = []
    models = sorted(df_merged['Method'].unique(), key=lambda x: (0 if 'xgboost' in x.lower() else 1, x))
    
    for model in models:
        m_rows = df_merged[df_merged['Method'] == model]
        modes = sorted(m_rows['Type'].unique())
        group_modes = []
        for mode in modes:
            mode_rows = m_rows[m_rows['Type'] == mode]
            label = mode
            if 'individual' in mode.lower() or 'x1y1' in mode.lower(): label = "Param"
            else:
                match = re.search(r"x(\d+)y(\d+)", mode)
                label = "Param (1/4)" if (match and match.group(1)=='2' and match.group(2)=='2') else f"Param (1/{int(match.group(1))*int(match.group(2))})" if match else mode
            
            # Filter out NaNs for jitter plot
            valid_rows = mode_rows.dropna(subset=['ratio'])
            group_modes.append({'mode_id': mode, 'clean_label': label, 'y': valid_rows['ratio'].values, 'z': valid_rows['passed'].values})
        plot_data.append({'model': model, 'modes': group_modes})

    fig, ax = plt.subplots(figsize=(12, 6), dpi=120)
    cmap = plt.cm.RdBu 
    norm = plt.Normalize(vmin=df_merged['passed'].min(), vmax=df_merged['passed'].max())
    x_cursor, x_ticks, x_labels = 0, [], []
    for group in plot_data:
        start_x = x_cursor
        for m_obj in group['modes']:
            if len(m_obj['y']) > 0:
                noise = np.random.uniform(-0.2, 0.2, size=len(m_obj['y']))
                ax.scatter(x_cursor + noise, m_obj['y'], c=m_obj['z'], cmap=cmap, norm=norm, s=25, alpha=0.7, edgecolors='grey', linewidth=0.3, zorder=2)
            x_ticks.append(x_cursor); x_labels.append(m_obj['clean_label']); x_cursor += 1
        ax.text((start_x + x_cursor - 1)/2, 1.45, group['model'].upper(), ha='center', va='bottom', fontweight='bold', fontsize=12, color='#333')
        ax.axvline(x=x_cursor - 0.5, color='#eee', linestyle='-', linewidth=1.5, zorder=0)
        x_cursor += 0.5
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, alpha=0.5, zorder=1)
    ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel(f"Ratio to Baseline ({metric})", fontsize=11)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, aspect=30, pad=0.01); cbar.set_label('Passed Events (Sample Size)', rotation=270, labelpad=15)
    plt.title(f"Performance Ratio ({metric}) - Color indicates Sample Size", loc='left', fontsize=12, pad=30)
    plt.ylim(0.5, 1.6); plt.tight_layout()
    return fig

# ==========================================
# 4. DASHBOARD CLASS
# ==========================================

class GridDashboard:
    def __init__(self):
        self.df = pd.DataFrame()
        self.df_cutflow = pd.DataFrame()
        self.expected_mx = []
        self.expected_my = []
        self.pivot_cache = {}
        self.fig_detail = None 
        
        self.btn_load = widgets.Button(description="Refresh Data", button_style='warning', icon='refresh')
        self.btn_save = widgets.Button(description="Save Plots", button_style='success', icon='save')
        self.out_status = widgets.Output()

        self.dd_metric = widgets.Dropdown(description="Metric:", options=[('AUC', 'auc'), ('Max SIC', 'max_sic'), ("Significance", "trafo_bin_sig"), ("Time", "time"), ('Cutflow: Passed', 'passed'), ('Cutflow: Eff %', 'eff')], value='auc')
        self.tog_compare = widgets.ToggleButtons(options=['Ratio', 'Pull'], description='Compare:', button_style='')
        self.cb_show_text = widgets.Checkbox(value=True, description='Values', indent=False, layout=widgets.Layout(width='auto'))
        self.cb_compact = widgets.Checkbox(value=False, description='Compact View', indent=False, layout=widgets.Layout(width='auto'))
        self.slider_font = widgets.IntSlider(value=10, min=6, max=18, description='Font:')
        self.slider_prec = widgets.IntSlider(value=3, min=0, max=5, description='Decimals:')
        self.dd_baseline = widgets.Dropdown(description="Baseline:")
        self.dd_mx = widgets.Dropdown(description="MX:")
        self.dd_my = widgets.Dropdown(description="MY:")
        
        self.out_heatmap = widgets.Output()
        self.out_details = widgets.Output()
        
        self.btn_load.on_click(self.load_data)
        self.btn_save.on_click(self.save_plots)
        for w in [self.dd_metric, self.dd_baseline, self.tog_compare, self.cb_show_text, self.cb_compact, self.slider_font, self.slider_prec]: w.observe(self.render_plots, names='value')
        self.dd_mx.observe(self.update_my_options, names='value')
        self.dd_my.observe(self.render_details, names='value')

        self.ui = widgets.VBox([
            widgets.HBox([self.btn_load, self.btn_save, self.dd_metric]),
            widgets.VBox([self.out_status], layout=widgets.Layout(border='1px solid #e0e0e0', padding='5px', margin='5px 0')),
            widgets.HTML("<hr>"), 
            widgets.HBox([self.dd_baseline, self.tog_compare, self.cb_compact, self.cb_show_text]),
            widgets.HBox([self.slider_font, self.slider_prec]), 
            self.out_heatmap,
            widgets.HTML("<hr><h3>Drill Down Inspector</h3>"), 
            widgets.HBox([self.dd_mx, self.dd_my]), 
            self.out_details
        ])
        
        self.load_data(None)

    def load_data(self, b):
        with self.out_status:
            clear_output()
            print("Initializing...")
            
            try:
                ref_mx, ref_my = load_reference_grid(JSON_GRID_PATH)
                self.df_cutflow = load_cutflow_data(CUTFLOW_PATH)
                
                tasks = []
                found_ind = glob.glob(os.path.join(BASE_PATH, "*", "individual", "**", "*.json"), recursive=True)
                for p in tqdm(found_ind, desc="Scanning 'Individual'", leave=False):
                    parts = p.split(os.sep)
                    if "individual" in parts: tasks.append((p, parts[parts.index("individual")-1], "individual"))

                found_param = glob.glob(os.path.join(BASE_PATH, "*", "parameterized", "**", "*.json"), recursive=True)
                for p in tqdm(found_param, desc="Scanning 'Parameterized'", leave=False):
                    parts = p.split(os.sep)
                    if "parameterized" in parts: tasks.append((p, parts[parts.index("parameterized")-1], "parameterized"))

                found_sparse = glob.glob(os.path.join(BASE_PATH, "*", "parametrized_reduce_*", "All", "*.json"))
                for p in tqdm(found_sparse, desc="Scanning 'Sparse'", leave=False):
                    parts = p.split(os.sep)
                    try:
                        config_folder = parts[-3]
                        f = re.search(r"factor_x_(\d+)_y_(\d+)", config_folder)
                        t = f"sparse_x{f.groups()[0]}y{f.groups()[1]}" if f else "sparse_param"
                        tasks.append((p, parts[-4], t))
                    except: continue

                total_files = len(tasks)
                results = []
                
                if total_files > 0:
                    with concurrent.futures.ProcessPoolExecutor(max_workers=4) as executor:
                        futures = [executor.submit(worker_process_json, t) for t in tasks]
                        pbar = tqdm(concurrent.futures.as_completed(futures), total=total_files, desc="Parsing Files (4 CPUs)", unit="file")
                        for future in pbar:
                            res = future.result()
                            if res: results.append(res)
                            time.sleep(0.001) 

                if results:
                    self.df = pd.DataFrame(results).sort_values('auc', ascending=False).drop_duplicates(subset=['ID', 'MX', 'MY'], keep='first')
                    self.expected_mx = sorted(list(set(ref_mx) | set(self.df['MX'].unique())))
                    self.expected_my = sorted(list(set(ref_my) | set(self.df['MY'].unique())))
                    self.precalculate_pivots()
                else:
                    self.df = pd.DataFrame(columns=['ID', 'MX', 'MY', 'auc'])
                
                if not self.df.empty:
                    methods = sorted(self.df['ID'].unique())
                    self.dd_baseline.options = methods
                    if methods: self.dd_baseline.value = next((m for m in methods if 'xgboost' in m.lower() and 'individual' in m.lower()), methods[0])
                    self.dd_mx.options = self.expected_mx
                    if self.expected_mx: self.dd_mx.value = self.expected_mx[0]; self.update_my_options(None)

                print(f"✅ Ready. Loaded {len(self.df)} records.")

            except Exception as e:
                print(f"❌ Error: {str(e)}")
        
        self.render_plots(None)

    def precalculate_pivots(self):
        self.pivot_cache = {}
        methods = self.df['ID'].unique()
        for met in ['auc', 'max_sic', 'trafo_bin_sig', 'time']:
            self.pivot_cache[met] = {}
            for m in methods:
                sub = self.df[self.df['ID'] == m]
                if sub.empty:
                    piv = pd.DataFrame(index=self.expected_my, columns=self.expected_mx)
                else:
                    piv = sub.pivot_table(index='MY', columns='MX', values=met, aggfunc='mean')
                    piv = piv.reindex(index=self.expected_my, columns=self.expected_mx).sort_index(ascending=False)
                self.pivot_cache[met][m] = piv

    def get_pivot(self, method, metric):
        if metric in self.pivot_cache and method in self.pivot_cache[metric]: return self.pivot_cache[metric][method]
        return pd.DataFrame()

    def save_plots(self, b):
        if not os.path.exists(PLOT_DIR): os.makedirs(PLOT_DIR)
        metric, base = self.dd_metric.value, self.dd_baseline.value
        methods = sorted(self.df['ID'].unique())
        
        with self.out_status:
            clear_output()
            print("Preparing workers...")
            p_base = self.get_pivot(base, metric)
            g_min, g_max = self.df[metric].min(), self.df[metric].max()
            job_args = []
            for m in methods:
                p_curr = self.get_pivot(m, metric)
                job_args.append((m, base, metric, p_curr, p_base, g_min, g_max, self.tog_compare.value, self.cb_show_text.value, self.slider_prec.value, PLOT_DIR))

            with concurrent.futures.ProcessPoolExecutor(max_workers=4) as executor:
                futures = [executor.submit(worker_save_plot, arg) for arg in job_args]
                pbar = tqdm(concurrent.futures.as_completed(futures), total=len(methods), desc="Saving Plots")
                for future in pbar:
                    future.result()
                    time.sleep(0.01)

            if self.fig_detail:
                mx, my = self.dd_mx.value, self.dd_my.value
                self.fig_detail.savefig(f"{PLOT_DIR}/detail_MX{mx}_MY{my}.png", bbox_inches='tight', dpi=100)
                print("Saved Detail Plot.")
            print(f"✅ All Done.")

    def update_my_options(self, change):
        self.dd_my.options = self.expected_my
        if self.expected_my: self.dd_my.value = self.expected_my[0]; self.render_details(None)

    def render_plots(self, change):
        if not self.expected_mx: return
        with self.out_heatmap:
            clear_output(wait=True)
            met = self.dd_metric.value
            if met in ['passed', 'eff']:
                if self.df_cutflow.empty: print("No Cutflow data."); return
                pivot = self.df_cutflow.pivot(index='MY', columns='MX', values=met).reindex(index=self.expected_my, columns=self.expected_mx).sort_index(ascending=False)
                fig, ax = plt.subplots(figsize=(10, 8), dpi=100)
                sns.heatmap(pivot, ax=ax, annot=self.cb_show_text.value, fmt=".0f" if met=='passed' else ".2f", cmap="viridis", cbar=True)
                plt.show()
                return

            if not self.dd_baseline.value: return
            base = self.dd_baseline.value
            methods = sorted(self.df['ID'].unique())
            n, is_compact = len(methods), self.cb_compact.value
            cols, width = (1, 8) if is_compact else (2, 16)
            
            fig, axes = plt.subplots(n, cols, figsize=(width, 5*n), dpi=100)
            if n == 1 and cols == 1: axes = np.array([axes])
            elif n == 1 or cols == 1: axes = axes.reshape(n, cols)
            
            p_base = self.get_pivot(base, met)
            g_min, g_max = self.df[met].min(), self.df[met].max()
            
            # Use copy of cmap to set bad values (NaN) to white/transparent
            cmap_val = plt.cm.PuBu.copy()
            cmap_val.set_bad(color='white')
            
            for i, m in enumerate(methods):
                p_curr = self.get_pivot(m, met)
                if is_compact:
                    self.plot_comparison_heatmap(axes[i, 0], p_curr, p_base, i==n-1)
                    axes[i, 0].set_ylabel(m, fontweight='bold', fontsize=12)
                else:
                    sns.heatmap(p_curr, ax=axes[i,0], annot=self.cb_show_text.value, fmt=f".{self.slider_prec.value}f", cmap=cmap_val, vmin=g_min, vmax=g_max, cbar=True)
                    axes[i,0].set_title(f"{m}", fontweight='bold'); axes[i,0].set_ylabel("MY"); axes[i,0].set_xlabel("MX" if i==n-1 else "")
                    self.plot_comparison_heatmap(axes[i,1], p_curr, p_base, i==n-1)
            plt.tight_layout(); plt.show()

    def plot_comparison_heatmap(self, ax, p_curr, p_base, is_bottom):
        cmap_diff = plt.cm.RdBu_r.copy()
        cmap_diff.set_bad(color='white')
        
        if self.tog_compare.value == 'Ratio':
            # FIX: No fillna(0)
            ratio = (p_curr/p_base)
            sns.heatmap(ratio, ax=ax, annot=self.cb_show_text.value, fmt=f".{self.slider_prec.value}f", cmap=cmap_diff, center=1, vmin=0.5, vmax=1.5)
            ax.set_title("Ratio vs Base")
        else:
            diff = (p_curr - p_base)
            sns.heatmap(diff, ax=ax, annot=self.cb_show_text.value, fmt=".2f", cmap=cmap_diff, center=0)
            ax.set_title("Diff vs Base")
        ax.set_xlabel("MX" if is_bottom else "")
        if not self.cb_compact.value: ax.set_yticks([])

    def render_details(self, change):
        mx, my, base = self.dd_mx.value, self.dd_my.value, self.dd_baseline.value
        if not mx or not my: return
        with self.out_details:
            clear_output(wait=True)
            cut_val = "N/A"
            if not self.df_cutflow.empty:
                subset = self.df_cutflow[(self.df_cutflow['MX']==mx) & (self.df_cutflow['MY']==my)]
                if not subset.empty: cut_val = f"{int(subset.iloc[0]['passed'])} (Eff: {subset.iloc[0]['eff']}%)"
            display(widgets.HTML(f"""<div style="background-color: #e3f2fd; border-left: 6px solid #2196f3; padding: 10px; margin-bottom: 20px;"><h4 style="margin:0; color:#0d47a1;">Signal Point: MX-{mx} MY-{my}</h4>Passed Events: <b>{cut_val}</b></div>"""))
            plot_metric = 'max_sic' if 'sic' in self.dd_metric.value else 'auc' if 'auc' in self.dd_metric.value else 'trafo_bin_sig'
            plot_base = next((m for m in self.df['ID'].unique() if 'xgboost' in m.lower() and 'individual' in m.lower()), base)
            try:
                fig = plot_comparison_fancy(self.df, self.df_cutflow, mx, my, self.expected_mx, self.expected_my, metric=plot_metric, baseline_id=plot_base)
                self.fig_detail = fig; display(fig)
            except Exception as e: print(f"Could not render summary plot: {e}")

            html = "<table border='1' style='width:100%; margin-top:20px; border-collapse:collapse; text-align:center;'><tr><th>Method</th><th>AUC</th><th>Max SIC</th><th>Time</th></tr>"
            imgs = []
            for m in sorted(self.df['ID'].unique()):
                row = self.df[(self.df['ID']==m)&(self.df['MX']==mx)&(self.df['MY']==my)]
                bg = "#eafaf1" if m==base else "white"
                if row.empty: html += f"<tr style='background:{bg}'><td>{m}</td><td colspan='3' style='color:#aaa'>No Data</td></tr>"; continue
                r = row.iloc[0]
                html += f"<tr style='background:{bg}'><td>{m}</td><td>{r['auc']:.4f}</td><td>{r['max_sic']:.2f}</td><td>{r['trafo_bin_sig']:.2f}</td><td>{r['time']:.1f}</td></tr>"
                if r['png_path'] and os.path.exists(r['png_path']):
                    img_w = widgets.Image(value=open(r['png_path'], "rb").read(), format='png', width=300)
                    imgs.append(widgets.VBox([widgets.Label(m, style={'font-weight':'bold'}), img_w], layout=widgets.Layout(border='1px solid #eee', margin='5px')))
            html += "</table>"
            display(widgets.HTML(html))
            display(widgets.Box(imgs, layout=widgets.Layout(display='flex', flex_flow='row wrap')))

d = GridDashboard()
display(d.ui)